# Tag & Probe: the HLT_Ele27_WPTight_Gsf efficiency with coffea (solutions)

HLT_Ele27_WPTight_Gsf fires on an isolated electron with $p_T > 27$ GeV passing a tight online ID.
We measure its efficiency in 2016 data with Z → ee events, using the tag & probe method.

**Tag** electron:
- $p_T > 27$ GeV and $|\eta| < 2.5$
- not in the ECAL barrel-endcap transition region $1.4442 < |\eta| < 1.566$
- passes the tight cut-based ID
- matched ($\Delta R < 0.1$) to an HLT electron that fired HLT_Ele27_WPTight_Gsf

**Probe** electron:
- opposite charge to the tag
- passes the same tight cut-based ID
- the tag-probe invariant mass is less than 30 GeV away from the Z mass (91.1876 GeV)
- $|\eta| < 2.5$ and not in the transition region, like the tag

A probe is **passing** if it is also matched ($\Delta R < 0.1$) to an HLT electron that fired HLT_Ele27_WPTight_Gsf,
and **failing** otherwise. In each probe $p_T$ bin:

$$\text{efficiency} = \frac{\text{passing}}{\text{passing} + \text{failing}}$$

All the functions are filled in here. Run all the cells to get the efficiency plot.

## Setup
Imports, the input file and the probe $p_T$ bin edges. To run on another file, change `FILENAME`.

In [ ]:
import awkward as ak
import hist
import matplotlib.pyplot as plt
import numpy as np
from coffea.nanoevents import NanoEventsFactory
from hist import intervals

FILENAME = "/data/mandi26/cms-opendata-singleelectron-small.root"

# Probe pT bin edges in GeV
PT_BINS = [5, 10, 12, 14, 16, 18, 20, 23, 26, 28, 30, 32, 34, 36, 38, 40, 45, 50, 60, 80, 100, 150, 250, 400]

## Step 1: Preselect the events
The tag has to fire the trigger, so we keep only the events that fired HLT_Ele27_WPTight_Gsf. A tag-probe pair also needs at least two electrons.

In [ ]:
def select_events(events):
    """Keep only the events that fired HLT_Ele27_WPTight_Gsf and have at least two electrons."""
    fired_trigger = events.HLT.Ele27_WPTight_Gsf
    has_two_electrons = ak.num(events.Electron) >= 2
    return events[fired_trigger & has_two_electrons]

## Step 2: Match electrons to the trigger
`TrigObj` holds the objects that the HLT built online. An offline electron is trigger matched if an HLT electron that passed HLT_Ele27_WPTight_Gsf is close to it. The last cell runs it on all the electrons and stores the result in two electron fields: `fires_tag_trigger`, which the tag must have, and `fires_probe_trigger`, which makes a probe pass. Here both are HLT_Ele27_WPTight_Gsf; to measure another trigger, only `fires_probe_trigger` changes.

In [ ]:
def trigger_match(electrons, trigobjs):
    """Return True for each electron matched to an HLT_Ele27_WPTight_Gsf trigger object.

    Use only trigger objects that are electrons (|id| == 11), have pt > 27 GeV and
    have bit 1 of filterBits set ("1e (WPTight)").
    An electron is matched if at least one of them is within dR < 0.1.
    The result has one True/False per electron, the same shape as `electrons`.

    Hint: electrons.metric_table(objects) gives the dR between every electron and every object.
    """
    is_electron = abs(trigobjs.id) == 11
    above_threshold = trigobjs.pt > 27
    passed_wptight = (trigobjs.filterBits & 2) != 0
    hlt_electrons = trigobjs[is_electron & above_threshold & passed_wptight]

    dr = electrons.metric_table(hlt_electrons)  # events x electrons x HLT electrons
    return ak.any(dr < 0.1, axis=2)

## Step 3: Make tag-probe pairs
Each Z → ee candidate is a pair of electrons: one is the tag, the other the probe. How you pair them is your choice, the rest of the code works with any of them. Can both electrons of an event be a probe?

In [ ]:
def make_pairs(electrons):
    """Build tag-probe pairs out of the electrons.

    Return an awkward array of records with the fields "tag" and "probe".
    Tag and probe must be different electrons. How you pair them is up to you,
    the rest of the code works for any choice.
    """
    # Every ordered pair (i, j) with i != j, so each electron is tried as a tag and as a probe.
    ij = ak.argcartesian({"tag": electrons, "probe": electrons})
    ij = ij[ij.tag != ij.probe]
    return ak.zip({"tag": electrons[ij.tag], "probe": electrons[ij.probe]})

### Other ways to make the pairs
Any of these can replace `make_pairs`, the rest of the code runs unchanged. They give different numbers of probes. To try one, run e.g. `make_pairs = make_pairs_combinations` before the last cell.

In [ ]:
def make_pairs_combinations(electrons):
    """Each pair of electrons once. The first electron (the one with higher pT) is the tag."""
    return ak.combinations(electrons, 2, fields=["tag", "probe"])


def make_pairs_combinations_both_ways(electrons):
    """Each pair of electrons in both orderings: the tag-probe pairs from ak.combinations and
    the same pairs as probe-tag, concatenated in each event."""
    tag_probe = ak.combinations(electrons, 2, fields=["tag", "probe"])
    probe_tag = ak.zip({"tag": tag_probe.probe, "probe": tag_probe.tag})
    return ak.concatenate([tag_probe, probe_tag], axis=1)


def make_pairs_exactly_two(electrons):
    """Only events with exactly two electrons. The first one is the tag, the second the probe."""
    electrons = electrons[ak.num(electrons) == 2]
    return ak.zip({"tag": electrons[:, 0], "probe": electrons[:, 1]})


def make_pairs_exactly_two_both_ways(electrons):
    """Only events with exactly two electrons. Each of them is tried as a tag and as a probe."""
    electrons = electrons[ak.num(electrons) == 2]
    first, second = electrons[:, 0], electrons[:, 1]
    return ak.concatenate([ak.zip({"tag": first, "probe": second}), ak.zip({"tag": second, "probe": first})])

## Step 4: Select the tags
The tag has to be a very clean electron, so that the pair really comes from a Z. Your function gets the tags (`pairs.tag`) and returns True/False for each of them.

In [ ]:
def select_tags(tags):
    """Return True for the electrons that pass the tag selection.

    pT > 27 GeV, |eta| < 2.5, not in 1.4442 < |eta| < 1.566, tight cut-based ID and firing
    the tag trigger (the "fires_tag_trigger" field, added to the electrons before making the pairs).
    """
    pass_pt = tags.pt > 27
    pass_eta = abs(tags.eta) < 2.5
    not_in_transition = (abs(tags.eta) < 1.4442) | (abs(tags.eta) > 1.566)
    pass_tight_id = tags.cutBased == 4
    return pass_pt & pass_eta & not_in_transition & pass_tight_id & tags.fires_tag_trigger

## Step 5: Select the probes
The probe passes the same offline ID as the tag, but has no trigger requirement: the trigger is what we measure.

In [ ]:
def select_probes(probes):
    """Return True for the electrons that pass the probe selection.

    Tight cut-based ID, |eta| < 2.5 and not in 1.4442 < |eta| < 1.566.
    No pT cut: we measure the efficiency as a function of the probe pT.
    """
    pass_eta = abs(probes.eta) < 2.5
    not_in_transition = (abs(probes.eta) < 1.4442) | (abs(probes.eta) > 1.566)
    pass_tight_id = probes.cutBased == 4
    return pass_eta & not_in_transition & pass_tight_id

## Step 6: Keep the Z candidates
A pair with opposite charges and a mass close to the Z mass is very likely a real Z → ee decay, so its probe is very likely a real electron.

In [ ]:
def select_z_candidates(pairs):
    """Return True for the tag-probe pairs that look like a Z -> ee decay.

    Tag and probe have opposite charge and their invariant mass is less than 30 GeV
    away from the Z mass (91.1876 GeV).
    """
    opposite_charge = pairs.tag.charge != pairs.probe.charge
    mass = (pairs.tag + pairs.probe).mass
    in_mass_window = abs(mass - 91.1876) < 30
    return opposite_charge & in_mass_window

## Step 7: Fill the histograms
Split the probes into passing and failing and histogram their $p_T$.

In [ ]:
def fill_histograms(pairs):
    """Fill one histogram of the probe pT with the passing probes and one with the failing probes.

    A probe passes if it fires the trigger we measure (its "fires_probe_trigger" field).
    Bin the histograms in PT_BINS. This must work whatever shape make_pairs returns.
    Return the two hist.Hist histograms (h_pass, h_fail).
    """
    probe_pt = ak.flatten(pairs.probe.pt, axis=None)
    is_passing = ak.flatten(pairs.probe.fires_probe_trigger, axis=None)

    h_pass = hist.Hist(hist.axis.Variable(PT_BINS, name="pt", label="Probe $p_T$ [GeV]"))
    h_fail = hist.Hist(hist.axis.Variable(PT_BINS, name="pt", label="Probe $p_T$ [GeV]"))
    h_pass.fill(pt=probe_pt[is_passing])
    h_fail.fill(pt=probe_pt[~is_passing])
    return h_pass, h_fail

## Step 8: Compute the efficiency
This one is given, just run the cell. The uncertainty is the Clopper-Pearson interval, the usual choice for an efficiency.

In [ ]:
def compute_efficiency(h_pass, h_fail):
    """Return the efficiency passing / (passing + failing) in each bin and its uncertainty.

    This function is given, there is nothing to fill in. The uncertainty is the Clopper-Pearson
    interval, returned as the (lower, upper) errors.
    """
    passing = h_pass.values()
    total = h_pass.values() + h_fail.values()
    # An empty bin gives 0 / 0 = nan, which is simply not drawn
    with np.errstate(divide="ignore", invalid="ignore"):
        efficiency = passing / total
        errors = intervals.ratio_uncertainty(passing, total, uncertainty_type="efficiency")
    return efficiency, errors

## Step 9: Plot the efficiency
This one is given too, just run the cell.

In [ ]:
def plot_efficiency(bin_edges, efficiency, errors):
    """Plot the efficiency versus the probe pT, show it and save it as efficiency.png.

    This function is given, there is nothing to fill in. Each bin is drawn as one point at the
    bin center, with x error = half the bin width and y error = `errors`.
    """
    bin_edges = np.array(bin_edges)
    centers = (bin_edges[1:] + bin_edges[:-1]) / 2
    half_widths = (bin_edges[1:] - bin_edges[:-1]) / 2

    fig, ax = plt.subplots()
    ax.errorbar(centers, efficiency, xerr=half_widths, yerr=errors, fmt="o", label="HLT_Ele27_WPTight_Gsf")
    ax.set_xscale("log")
    ax.set_xlim(5, 400)
    ax.set_ylim(0, 1.2)
    ax.set_xticks([5, 10, 20, 30, 50, 100, 200, 400], ["5", "10", "20", "30", "50", "100", "200", "400"])
    ax.set_xlabel("Offline electron $p_T$ [GeV]")
    ax.set_ylabel("L1T + HLT efficiency")
    ax.legend(loc="lower right")
    fig.savefig("efficiency.png")
    plt.show()

## Run the measurement
This cell loads the events and puts all the steps together. Run it once all the functions above are filled in.

In [ ]:
# Open the file in virtual mode: a branch is only read from the file the first time it is used
events = NanoEventsFactory.from_root({FILENAME: "Events"}, mode="virtual").events()
print("Events in the file:", len(events))

events = select_events(events)
print("Events after the preselection:", len(events))

# Trigger match every electron before making the pairs and store the results as electron fields,
# so tags and probes carry them along however make_pairs builds the pairs. The tag must fire the
# tag trigger, and a probe passes if it fires the trigger we measure. Here both are
# HLT_Ele27_WPTight_Gsf; to measure another trigger, only fires_probe_trigger changes.
events["Electron", "fires_tag_trigger"] = trigger_match(events.Electron, events.TrigObj)
events["Electron", "fires_probe_trigger"] = trigger_match(events.Electron, events.TrigObj)

pairs = make_pairs(events.Electron)
pairs = pairs[select_tags(pairs.tag)]
pairs = pairs[select_probes(pairs.probe)]
pairs = pairs[select_z_candidates(pairs)]
print("Tag-probe pairs:", ak.count(pairs.probe.pt, axis=None))

h_pass, h_fail = fill_histograms(pairs)
print("Passing probes:", int(h_pass.sum(flow=True)))
print("Failing probes:", int(h_fail.sum(flow=True)))

efficiency, errors = compute_efficiency(h_pass, h_fail)
plot_efficiency(h_pass.axes[0].edges, efficiency, errors)
print("Saved the plot to efficiency.png")

Different pairing choices in Step 3 give different numbers of probes. Do the efficiencies agree?

## Scaling up with a coffea processor
To run on many files, we put the steps of the last cell in a coffea processor. Its `process` method gets the events of one chunk of one file and returns the passing and failing histograms of its dataset. coffea adds up the outputs of all the chunks.

In [ ]:
from coffea import processor
from coffea.nanoevents import NanoAODSchema


class TagAndProbeProcessor(processor.ProcessorABC):
    def process(self, events):
        dataset = events.metadata["dataset"]

        events = select_events(events)
        events["Electron", "fires_tag_trigger"] = trigger_match(events.Electron, events.TrigObj)
        events["Electron", "fires_probe_trigger"] = trigger_match(events.Electron, events.TrigObj)

        pairs = make_pairs(events.Electron)
        pairs = pairs[select_tags(pairs.tag)]
        pairs = pairs[select_probes(pairs.probe)]
        pairs = pairs[select_z_candidates(pairs)]

        # Simulation only: keep pairs whose tag and probe are prompt electrons from the same Z (comment out to turn off)
        if events.metadata["is_mc"]:
            genmatch = (
                (pairs.tag.matched_gen.distinctParentIdxG == pairs.probe.matched_gen.distinctParentIdxG)
                & (pairs.tag.matched_gen.distinctParent.pdgId == 23)
                & (pairs.probe.matched_gen.distinctParent.pdgId == 23)
            )
            genpartflav = (pairs.tag.genPartFlav == 1) & (pairs.probe.genPartFlav == 1)
            pairs = pairs[ak.fill_none(genmatch, False) & genpartflav]

        h_pass, h_fail = fill_histograms(pairs)
        return {dataset: {"pass": h_pass, "fail": h_fail}}

    def postprocess(self, accumulator):
        return accumulator

## The fileset
Two datasets: the 2016 SingleElectron data and the Drell-Yan (Z → ee) simulation. The metadata of each dataset reaches the processor in `events.metadata`.

In [ ]:
fileset = {
    "Data": {
        "files": [
            "/data/mandi26/cms-opendata-singleelectron-small.root",
            "/data/mandi26/cms-opendata-singleelectron-1.root",
            "/data/mandi26/cms-opendata-singleelectron-2.root",
        ],
        "metadata": {"is_mc": False},
    },
    "DY": {
        "files": [
            "/data/mandi26/cms-opendata-dytollamcatnlofxfx-1.root",
            "/data/mandi26/cms-opendata-dytollamcatnlofxfx-2.root",
        ],
        "metadata": {"is_mc": True},
    },
}

## Run the processor
The runner splits the files into chunks and processes them in parallel with 4 worker processes.

In [ ]:
runner = processor.Runner(executor=processor.FuturesExecutor(workers=4), schema=NanoAODSchema)
output = runner(fileset, processor_instance=TagAndProbeProcessor(), treename="Events")
output

## Data and simulation efficiencies

In [ ]:
eff_data, err_data = compute_efficiency(output["Data"]["pass"], output["Data"]["fail"])
eff_mc, err_mc = compute_efficiency(output["DY"]["pass"], output["DY"]["fail"])

bin_edges = np.array(PT_BINS)
centers = (bin_edges[1:] + bin_edges[:-1]) / 2
half_widths = (bin_edges[1:] - bin_edges[:-1]) / 2

fig, ax = plt.subplots()
ax.errorbar(centers, eff_data, xerr=half_widths, yerr=err_data, fmt="o", label="Data")
ax.errorbar(centers, eff_mc, xerr=half_widths, yerr=err_mc, fmt="o", label="DY simulation")
ax.set_xscale("log")
ax.set_xlim(5, 400)
ax.set_ylim(0, 1.2)
ax.set_xticks([5, 10, 20, 30, 50, 100, 200, 400], ["5", "10", "20", "30", "50", "100", "200", "400"])
ax.set_xlabel("Offline electron $p_T$ [GeV]")
ax.set_ylabel("L1T + HLT efficiency")
ax.legend(loc="lower right", title="HLT_Ele27_WPTight_Gsf")
plt.show()

## Scale factors
The scale factor is the ratio of the data and simulation efficiencies, $SF = \varepsilon_{data} / \varepsilon_{MC}$. For a ratio, the errors add in quadrature:

$$\sigma_{SF}^2 = \left(\frac{\sigma_{data}}{\varepsilon_{MC}}\right)^2 + \left(\frac{\varepsilon_{data}\,\sigma_{MC}}{\varepsilon_{MC}^2}\right)^2$$

The efficiency errors are asymmetric. The SF goes up when the data efficiency goes up or the simulation efficiency goes down, so the upper SF error uses the upper data error and the lower simulation error, and the other way around for the lower SF error.

In [ ]:
# err_data and err_mc hold the (lower, upper) errors of the efficiencies
with np.errstate(divide="ignore", invalid="ignore"):
    sf = eff_data / eff_mc
    sf_lower = np.sqrt((err_data[0] / eff_mc) ** 2 + (eff_data * err_mc[1] / eff_mc**2) ** 2)
    sf_upper = np.sqrt((err_data[1] / eff_mc) ** 2 + (eff_data * err_mc[0] / eff_mc**2) ** 2)

# 0 / 0 gives nan (no passing probes in data and simulation, or no probes at all): set the SF to 1 there
sf[np.isnan(sf)] = 1

fig, ax = plt.subplots()
ax.errorbar(centers, sf, xerr=half_widths, yerr=[sf_lower, sf_upper], fmt="o")
ax.axhline(1, color="k", linestyle="--", linewidth=1)
ax.set_xscale("log")
ax.set_xlim(5, 400)
ax.set_xticks([5, 10, 20, 30, 50, 100, 200, 400], ["5", "10", "20", "30", "50", "100", "200", "400"])
ax.set_xlabel("Offline electron $p_T$ [GeV]")
ax.set_ylabel("Data / simulation scale factor")
plt.show()

sf

## Closure test
We apply the scale factors to the probes of an independent DY file. In each $p_T$ bin, every passing probe gets the weight $SF = \varepsilon_{data} / \varepsilon_{MC}$ and every failing probe the weight $(1 - \varepsilon_{data}) / (1 - \varepsilon_{MC})$. The corrected simulation efficiency should then agree with data within the statistical uncertainties. This only checks the scale factors: in an analysis they are applied to the simulation as event weights.

In [ ]:
events_closure = NanoEventsFactory.from_root(
    {"/data/mandi26/cms-opendata-dytollamcatnlofxfx-3.root": "Events"},
    mode="virtual",
    metadata={"dataset": "DY", "is_mc": True},
).events()
output_closure = TagAndProbeProcessor().process(events_closure)
output_closure

In [ ]:
passing = output_closure["DY"]["pass"].values()
failing = output_closure["DY"]["fail"].values()
eff_closure, err_closure = compute_efficiency(output_closure["DY"]["pass"], output_closure["DY"]["fail"])

# Weight of the failing probes in each pT bin; 0 / 0 is set to 1, like the SF
with np.errstate(divide="ignore", invalid="ignore"):
    weight_fail = (1 - eff_data) / (1 - eff_mc)
weight_fail[np.isnan(weight_fail)] = 1

with np.errstate(divide="ignore", invalid="ignore"):
    eff_corrected = sf * passing / (sf * passing + weight_fail * failing)
    # Same relative uncertainty as the uncorrected efficiency of this file
    err_corrected = err_closure * eff_corrected / eff_closure

fig, ax = plt.subplots()
ax.errorbar(centers, eff_data, xerr=half_widths, yerr=err_data, fmt="o", label="Data")
ax.errorbar(centers, eff_closure, xerr=half_widths, yerr=err_closure, fmt="o", label="DY simulation")
ax.errorbar(centers, eff_corrected, xerr=half_widths, yerr=err_corrected, fmt="o", label="DY simulation, corrected")
ax.set_xscale("log")
ax.set_xlim(5, 400)
ax.set_ylim(0, 1.2)
ax.set_xticks([5, 10, 20, 30, 50, 100, 200, 400], ["5", "10", "20", "30", "50", "100", "200", "400"])
ax.set_xlabel("Offline electron $p_T$ [GeV]")
ax.set_ylabel("L1T + HLT efficiency")
ax.legend(loc="lower right", title="HLT_Ele27_WPTight_Gsf")
plt.show()